# 1.5 数据处理管道工程化 (Data Pipeline Engineering)

> 🕐 预估学习时间：35分钟

大规模训练中，数据处理管道是决定训练效率的关键瓶颈之一。本节从工程角度讨论如何构建高效的数据处理管道。

本节涵盖：
- 数据加载性能优化（DataLoader 调优）
- 流式数据处理（Streaming）
- 数据版本管理
- 数据质量监控管道
- 分布式数据处理

## 1. DataLoader 性能优化

**目的**：最大化 GPU 利用率，避免数据加载成为训练瓶颈

**关键优化技巧**：
- **多进程加载**：`num_workers=4~8`，并行预处理数据
- **预取（prefetch）**：提前准备下一批数据，GPU 不等 I/O
- **固定内存（pin_memory）**：加速 CPU→GPU 数据传输（约 2x）
- **批量打包**：将短序列拼接为定长批次，减少 padding 浪费
- **内存映射（Memory Mapping）**：用 `np.memmap` 避免重复读磁盘

**PyTorch DataLoader 配置参考**：
```python
DataLoader(
    dataset,
    batch_size=32,
    num_workers=4,          # 多进程预处理
    pin_memory=True,        # 加速 CPU→GPU 传输
    prefetch_factor=2,      # 每个 worker 预取 batch 数
    persistent_workers=True # 进程常驻，避免反复创建
)
```

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import time

torch.manual_seed(42)

class DummyTextDataset(Dataset):
    def __init__(self, n_samples=100000, seq_len=512, vocab_size=32000):
        self.data = torch.randint(0, vocab_size, (n_samples, seq_len))
        self.labels = torch.randint(0, vocab_size, (n_samples, seq_len))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

dataset = DummyTextDataset(n_samples=10000)

def benchmark_dataloader(ds, **kwargs):
    loader = DataLoader(ds, batch_size=64, **kwargs)
    start = time.time()
    for batch_idx, (x, y) in enumerate(loader):
        if batch_idx >= 50:
            break
    elapsed = time.time() - start
    return elapsed, 50 * 64 / elapsed

configs = [
    {'num_workers': 0, 'pin_memory': False},
    {'num_workers': 4, 'pin_memory': False},
    {'num_workers': 4, 'pin_memory': True},
    {'num_workers': 4, 'pin_memory': True, 'prefetch_factor': 4},
]

print('=== DataLoader Performance ===')
print(f'{"Config":<45} {"Time":>8} {"Throughput":>12}')
print('-' * 67)

for cfg in configs:
    elapsed, throughput = benchmark_dataloader(dataset, **cfg)
    desc = f'workers={cfg["num_workers"]}, pin={cfg["pin_memory"]}'
    if 'prefetch_factor' in cfg:
        desc += f', prefetch={cfg["prefetch_factor"]}'
    print(f'{desc:<45} {elapsed:>7.2f}s {throughput:>8.0f} samp/s')

print(f'\nKey: num_workers > 1 is essential for GPU training.')
print(f'pin_memory=True accelerates CPU→GPU transfer via pinned memory.')
print(f'prefetch_factor prepares next batches while GPU computes.')

## 2. 流式数据处理（Streaming）

**目的**：处理超过内存容量的大规模数据集

**当数据集无法全部装入内存时**：
- HuggingFace `datasets` 库的 streaming 模式
- Apache Arrow 的零拷贝列式存储
- 惰性加载（Lazy Loading）只加载当前需要的分片

**Streaming Pipeline 架构**：
```
原始数据 → 分片存储(Shard) → Stream Reader → 预处理 → Tokenize → Batch → GPU
```

**Mosaic StreamingDataset**（训练加速专用）：
- 支持流式读取 + 在线 tokenization
- 确定性 shuffle：可恢复训练
- 多节点分布式数据分发

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader
import random

class StreamingTextDataset(IterableDataset):
    def __init__(self, n_shards=10, seq_len=512, vocab_size=32000):
        self.n_shards = n_shards
        self.seq_len = seq_len
        self.vocab_size = vocab_size

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is None:
            shards = list(range(self.n_shards))
        else:
            per_worker = self.n_shards // worker_info.num_workers
            worker_id = worker_info.id
            shards = list(range(worker_id * per_worker, (worker_id + 1) * per_worker))

        for shard_idx in shards:
            g = torch.Generator()
            g.manual_seed(shard_idx)
            data = torch.randint(0, self.vocab_size, (1000, self.seq_len), generator=g)
            labels = torch.randint(0, self.vocab_size, (1000, self.seq_len), generator=g)
            for i in range(len(data)):
                yield data[i], labels[i]

dataset = StreamingTextDataset(n_shards=10)
loader = DataLoader(dataset, batch_size=64, num_workers=2)

print('=== Streaming Data Pipeline ===')
samples_seen = 0
for batch_idx, (x, y) in enumerate(loader):
    samples_seen += len(x)
    if batch_idx < 3:
        print(f'  Batch {batch_idx}: x={x.shape}, y={y.shape}')
    if batch_idx >= 15:
        break

print(f'\nTotal batches: {batch_idx+1}, Total samples: {samples_seen:,}')
print(f'\nKey: Streaming datasets never load all data into memory at once.')
print(f'Each worker processes independent shards for parallelism.')
print(f'HuggingFace datasets.load_dataset(..., streaming=True) follows this pattern.')

## 3. 数据版本管理

**为什么需要版本管理**：
- 训练数据是模型的核心资产，数据变更需可追溯
- 复现实验需要确切知道使用了哪个版本的数据
- 多团队协作需要统一的数据视图

**工具选型**：
| 工具 | 适用规模 | 特点 |
|------|----------|------|
| DVC | 中型数据集 | Git-like 接口，与 Git 集成 |
| LakeFS | 大型数据集 | 数据湖版本控制，分支操作 |
| HuggingFace Datasets | 训练用数据集 | 内置版本控制和缓存 |
| Delta Lake | 超大规模 | Spark 生态，ACID 事务 |

In [ ]:
import hashlib
import json
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class DatasetVersion:
    version_id: str
    created_at: str
    n_samples: int
    schema: dict
    preprocessing_steps: list
    checksum: str
    parent_version: str = None

class DataVersionManager:
    def __init__(self):
        self.versions = {}

    def create_version(self, data, preprocessing_steps=None, parent=None):
        content_hash = hashlib.md5(str(data.shape).encode()).hexdigest()[:8]
        version_id = f'v{len(self.versions) + 1}_{content_hash}'
        version = DatasetVersion(
            version_id=version_id,
            created_at=datetime.now().isoformat(),
            n_samples=len(data),
            schema={'shape': list(data.shape), 'dtype': str(data.dtype)},
            preprocessing_steps=preprocessing_steps or [],
            checksum=content_hash,
            parent_version=parent
        )
        self.versions[version_id] = version
        return version_id

    def get_lineage(self, version_id):
        lineage = []
        vid = version_id
        while vid:
            v = self.versions[vid]
            lineage.append(vid)
            vid = v.parent_version
        return lineage[::-1]

manager = DataVersionManager()
raw_data = torch.randn(100000, 128)
v1 = manager.create_version(raw_data, ['raw_ingestion', 'format_normalization'])
deduped = raw_data[torch.randperm(len(raw_data))[:80000]]
v2 = manager.create_version(deduped, ['deduplication', 'quality_filter'], parent=v1)
filtered = deduped[:60000]
v3 = manager.create_version(filtered, ['pii_removal', 'tokenization'], parent=v2)

print('=== Data Version Management ===')
for vid in [v1, v2, v3]:
    v = manager.versions[vid]
    print(f'\n  {vid}:')
    print(f'    Samples: {v.n_samples:,}')
    print(f'    Steps: {v.preprocessing_steps}')
    print(f'    Parent: {v.parent_version}')

print(f'\nLineage of {v3}: {" -> ".join(manager.get_lineage(v3))}')
print(f'\nKey: Track data lineage for reproducibility and debugging.')
print(f'Each preprocessing step creates a new immutable version.')

## 4. 数据质量监控管道

**生产环境中的数据质量指标**：
- **覆盖率监控**：各领域数据占比变化
- **分布漂移**：数据分布的 KL 散度/JS 散度
- **异常检测**：空文本、乱码、超长文本比例
- **新鲜度**：数据时间戳分布

**监控架构**：
```
数据管道 → 指标采集 → 时序存储(Prometheus/InfluxDB) → 仪表盘(Grafana) → 告警
```

In [ ]:
import torch
import torch.nn.functional as F
from collections import defaultdict

class DataQualityMonitor:
    def __init__(self, alert_thresholds=None):
        self.metrics = defaultdict(list)
        self.alerts = []
        self.thresholds = alert_thresholds or {
            'empty_ratio': 0.01,
            'max_length': 100000,
            'duplicate_ratio': 0.05,
            'lang_mismatch': 0.03,
        }

    def check_batch(self, texts, languages=None):
        batch_metrics = {
            'n_samples': len(texts),
            'empty_ratio': sum(1 for t in texts if len(t.strip()) == 0) / max(len(texts), 1),
            'avg_length': sum(len(t) for t in texts) / max(len(texts), 1),
            'max_length': max((len(t) for t in texts), default=0),
        }

        alerts = []
        if batch_metrics['empty_ratio'] > self.thresholds['empty_ratio']:
            alerts.append(f'Empty ratio {batch_metrics["empty_ratio"]:.2%} > {self.thresholds["empty_ratio"]:.2%}')
        if batch_metrics['max_length'] > self.thresholds['max_length']:
            alerts.append(f'Max length {batch_metrics["max_length"]} > {self.thresholds["max_length"]}')

        for k, v in batch_metrics.items():
            if isinstance(v, (int, float)):
                self.metrics[k].append(v)
        self.alerts.extend(alerts)
        return batch_metrics, alerts

    def summary(self):
        return {
            k: {'mean': sum(v)/len(v), 'min': min(v), 'max': max(v)}
            for k, v in self.metrics.items() if v
        }

monitor = DataQualityMonitor()

batches = [
    ['Hello world ' * 10, 'Short', 'Medium length text here.'] * 10,
    ['', 'Valid text', 'Valid text'] * 10,
    ['Normal content ' * 20, 'Good data here.'] * 10,
]

print('=== Data Quality Monitoring ===')
for i, batch in enumerate(batches):
    metrics, alerts = monitor.check_batch(batch)
    print(f'\nBatch {i+1}:')
    print(f'  Samples: {metrics["n_samples"]}')
    print(f'  Avg length: {metrics["avg_length"]:.1f}')
    print(f'  Empty ratio: {metrics["empty_ratio"]:.1%}')
    if alerts:
        for alert in alerts:
            print(f'  ALERT: {alert}')

print(f'\nOverall summary: {json.dumps(monitor.summary(), indent=2)}')
print(f'Total alerts: {len(monitor.alerts)}')
print(f'\nKey: Continuous data quality monitoring prevents garbage-in-garbage-out.')
print(f'Production pipelines should export metrics to Prometheus/Grafana.')

## 5. 分布式数据处理

**当数据量达到 TB 级时**：
- **Spark/Ray Data**：分布式 ETL 处理
- **数据分片（Sharding）**：按文件或记录分片
- **数据本地性**：尽量在数据所在的机器上训练
- **预 tokenize + 缓存**：避免每次重复 tokenize

**产业最佳实践**：
```
原始数据(TB级) → Spark 清洗 → Parquet 分片 → 预 Tokenize → Arrow 格式 → 分布式训练读取
```

**常用格式对比**：
| 格式 | 读取速度 | 压缩率 | 适用场景 |
|------|----------|--------|----------|
| JSONL | 慢 | 低 | 小规模原型 |
| Parquet | 快 | 高 | 大规模列式数据 |
| Arrow | 最快 | 中 | 训练中频繁读取 |
| TFRecord | 快 | 高 | TensorFlow 生态 |

In [ ]:
import torch
import math

def estimate_pipeline_throughput(
    data_size_tb=10,
    tokenize_speed_gb_per_sec=0.5,
    gpu_compute_tflops=312,
    model_flops_per_token=6e12
):
    tokenize_time = data_size_tb * 1000 / tokenize_speed_gb_per_sec / 3600
    train_throughput = gpu_compute_tflops * 1e12 * 0.5 / model_flops_per_token
    train_time_per_epoch = data_size_tb * 1e12 / 2 / train_throughput / 3600

    print('=== Pipeline Throughput Estimation ===')
    print(f'Data: {data_size_tb} TB')
    print(f'Tokenization speed: {tokenize_speed_gb_per_sec} GB/s')
    print(f'\nTokenization time: ~{tokenize_time:.1f} hours')
    print(f'Training throughput: ~{train_throughput:.0f} tokens/s')
    print(f'Training time/epoch: ~{train_time_per_epoch:.1f} hours')

    if tokenize_time > train_time_per_epoch:
        print(f'\nWARNING: Tokenization ({tokenize_time:.1f}h) > Training ({train_time_per_epoch:.1f}h)')
        print(f'Suggestion: Pre-tokenize and cache all data before training.')
    else:
        print(f'\nPipeline is compute-bound (training dominates).')
        print(f'Suggestion: Increase GPU count to reduce training time.')

estimate_pipeline_throughput()

print(f'\nKey: Pre-tokenization is essential when tokenization speed < training speed.')
print(f'Store pre-tokenized data in Arrow/Parquet format for efficient I/O.')

## 📝 课后思考题

1. 当 `num_workers` 设置过大时可能出现什么问题？（提示：CPU 内存、进程间通信）
2. Streaming Dataset 的随机 shuffle 如何实现？与内存 shuffle 有何不同？
3. 如何设计数据质量监控的告警阈值？过高和过低各有什么问题？
4. 在 GPU 集群上训练时，如何避免数据加载成为瓶颈？